<a href="https://colab.research.google.com/github/VictorNevola/ml-study/blob/main/ml_11_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("diabetes_en.csv")
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print(f"Single tree  -> test accuracy: {tree.score(X_test, y_test):.3f}")

Single tree  -> test accuracy: 0.727


In [2]:
forest = RandomForestClassifier(n_estimators=200, random_state=42)
forest.fit(X_train, y_train)
print(f"Random Forest -> test accuracy: {forest.score(X_test, y_test):.3f}")

Random Forest -> test accuracy: 0.747


In [3]:
# take one test patient and see how the individual trees vote
patient = X_test.iloc[[7]]
true_label = y_test.iloc[7]

votes = np.array([t.predict(patient.values)[0] for t in forest.estimators_])
n_diabetes = int(votes.sum())
n_no = int((votes == 0).sum())

print(f"True label: {true_label}")
print(f"Trees voting 'diabetes' (1): {n_diabetes}")
print(f"Trees voting 'no'       (0): {n_no}")
print(f"Forest decision (majority): {forest.predict(patient)[0]}")
print(f"Probability = {n_diabetes}/{len(votes)} = {forest.predict_proba(patient)[0][1]:.0%}")



True label: 1
Trees voting 'diabetes' (1): 139
Trees voting 'no'       (0): 61
Forest decision (majority): 1
Probability = 139/200 = 70%


In [4]:

individual_acc = [
    (t.predict(X_test.values) == y_test.values).mean()
    for t in forest.estimators_
]

print(f"Individual trees -> mean accuracy: {np.mean(individual_acc):.3f}")
print(f"                    worst / best : {min(individual_acc):.3f} / {max(individual_acc):.3f}")
print(f"\nThe FOREST (all together): {forest.score(X_test, y_test):.3f}")
print("\n=> The forest beats basically every single tree on its own.")

Individual trees -> mean accuracy: 0.680
                    worst / best : 0.597 / 0.779

The FOREST (all together): 0.747

=> The forest beats basically every single tree on its own.


In [5]:

from sklearn.metrics import accuracy_score

def spread(make_model, n=15):
    accs = []
    for seed in range(n):
        Xa, Xb, ya, yb = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        m = make_model(seed).fit(Xa, ya)
        accs.append(accuracy_score(yb, m.predict(Xb)))
    return np.mean(accs), np.std(accs), min(accs), max(accs)

t_mean, t_std, t_min, t_max = spread(lambda s: DecisionTreeClassifier(random_state=s))
f_mean, f_std, f_min, f_max = spread(lambda s: RandomForestClassifier(n_estimators=100, random_state=s))

print("Over 15 different splits:")
print(f"  Single tree   -> mean {t_mean:.3f} | std {t_std:.3f} | range [{t_min:.3f}, {t_max:.3f}]")
print(f"  Random Forest -> mean {f_mean:.3f} | std {f_std:.3f} | range [{f_min:.3f}, {f_max:.3f}]")



Over 15 different splits:
  Single tree   -> mean 0.697 | std 0.032 | range [0.630, 0.740]
  Random Forest -> mean 0.764 | std 0.033 | range [0.695, 0.812]


In [7]:

from sklearn.model_selection import cross_val_score

for n in [1, 5, 10, 50, 100, 300, 500, 900]:
    score = cross_val_score(
        RandomForestClassifier(n_estimators=n, random_state=42),
        X_train, y_train, cv=5
    ).mean()
    print(f"n_estimators={n:3d} -> CV accuracy {score:.3f}")


n_estimators=  1 -> CV accuracy 0.717
n_estimators=  5 -> CV accuracy 0.749
n_estimators= 10 -> CV accuracy 0.762
n_estimators= 50 -> CV accuracy 0.757
n_estimators=100 -> CV accuracy 0.766
n_estimators=300 -> CV accuracy 0.766
n_estimators=500 -> CV accuracy 0.764
n_estimators=900 -> CV accuracy 0.767
